### Data Reading


#####Data reading for json

In [0]:
df_json = spark.read.format('json').option('inferSchema',True)\
                    .option('header',True)\
                    .option('multiLine',True)\
                    .load('/Volumes/workspace/default/json/drivers.json')

                        

In [0]:
df_json.display()

#####Data reading for csv

In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load('/Volumes/workspace/default/data/BigMart Sales.csv')# Data uploaded by upload files to a volukme

In [0]:
# df.show()
# display(df)
df.count()

In [0]:
df2 = spark.table('workspace.default.big_mart_sales') # Data uploaded by create or modify table

###Schema Definition

In [0]:
df.printSchema()

#####1.DDL schema

In [0]:
#User-defined schema
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,       
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [0]:
df = spark.read.format('csv')\
          .schema(my_ddl_schema)\
          .option('header',True)\
          .load('/Volumes/workspace/default/data/BigMart Sales.csv')
df.count()
df.printSchema()
df.count()
df.show()

#####StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:

my_struct_schema = StructType([ 
                              StructField('Item_Identifier',StringType(),True), 
                              StructField('Item_Weight',StringType(),True), 
                              StructField('Item_Fat_Content',StringType(),True), 
                              StructField('Item_Visibility',StringType(),True), 
                              StructField('Item_Type',StringType(),True), 
                              StructField('Item_MRP',StringType(),True), 
                              StructField('Outlet_Identifier',StringType(),True),
                              StructField('Outlet_Establishment_Year',StringType(),True), 
                              StructField('Outlet_Size',StringType(),True), 
                              StructField('Outlet_Location_Type',StringType(),True), 
                              StructField('Outlet_Type',StringType(),True), 
                              StructField('Item_Outlet_Sales',StringType(),True)

])

In [0]:
df = spark.read.format('csv').schema(my_struct_schema).option('header',True).load('/Volumes/workspace/default/data/BigMart Sales.csv')
df.show()
df.printSchema()


###Select

In [0]:
df.display()

In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

###Alias

In [0]:
df.select(col('Item_Identifier').alias('Item_id')).display()

###Filter

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col("Item_Weight") < 10.0)).display()

In [0]:
df.filter((col("Outlet_Size").isNull())&(col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()

###withColumnRenamed

In [0]:
df.withColumnRenamed("Item_Weight","Item_wt").display()


###withColumn

In [0]:
from pyspark.sql.functions import *
#Create a new column using withCoulmn with contant value using lit()fn
df = df.withColumn("New_Column",lit(10))
df.display()


In [0]:
#new column
df.withColumn('multiply', col('Item_Weight')*col('Item_MRP')).display()


In [0]:
#Modification of column data
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Regular','reg',)) \
       .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Low Fat','LF')).display()

###Type Casting

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType())) 
df.printSchema()


#Sort/orderBy

In [0]:
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(col("Item_Visibility").asc()).display()

In [0]:
# df.sort(["Item_Visibility","Item_MRP"],ascending=[0,0]).display()
 # OR

from pyspark.sql.functions import *

df.sort(desc("Item_Visibility"), desc("Item_MRP")).display()

In [0]:
# df.sort(["Item_Visibility","Item_MRP"],ascending=[0,1]).display()
 # Or
from pyspark.sql.functions import *

df.sort(desc("Item_Visibility"), asc("Item_MRP")).display()


In [0]:
from pyspark.sql.functions import *

df.orderBy(col("Item_Visibility").desc(), col("Item_MRP").desc()).display()

###Limit

In [0]:
df.limit(10).display()

###Drop

In [0]:
#Drop columns
df.drop("Item_Visibility").display()

In [0]:
df.drop("Item_Visibility", "Item_Type").display()

###Drop_duplicates

In [0]:
df.drop_duplicates().display()


In [0]:
# df.drop_duplicates(["Item_Type"]).display()
#or
# df.dropDuplicates(["Item_Type"]).display()
#or
df.dropDuplicates(subset=["Item_Type"]).display()



In [0]:
df.distinct().display()

###Union and Union by name

#####Preparing dataframes

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()
     
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad','1',),
        ('sid','2',)]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)

df1.display()
     
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

###String Functions

#####Initcap(), upper(), lower()

In [0]:
df.select(initcap('Item_Type')).display()
df.select(lower('Item_Type')).display()
df.select(upper('Item_Type').alias('upper_Item_Type')).display()


###Date functions


#####Current_date

In [0]:
from pyspark.sql.functions import *
df = df.withColumn('curr_date',current_date())
df.display()

#####Date_add


In [0]:
df = df.withColumn('week_after',date_add('curr_date',7))

df.display()
     

#####Date sub

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()
#or
df = df.withColumn('week_before',date_add('curr_date',-7)) 

df.display()   

#####Date diff(intervals)

In [0]:
df = df.withColumn('datediff',datediff('week_after','curr_date'))

df.display()

#####Date formats

In [0]:
df = df.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))

df.display()

###Handling Nulls


#####Dropping nulls

In [0]:
df.dropna().display()

In [0]:
df.dropna('all').display()#Checks all null and drops
df.dropna('any').display() #checks null in any column and drops

In [0]:
df.dropna(subset=["Outlet_Size"]).display()

#####Filling nulls

In [0]:

df.fillna('NotAvailable').display()   

df.fillna('NotAvailable',subset=['Outlet_Size']).display()

In [0]:
df.fillna({'Outlet_Size':'NotAvailable'}).display()

###Split and indexing

In [0]:
#Split
from pyspark.sql.functions import *
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()
     



In [0]:
#Indexing
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

###Explode

In [0]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()   

df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()
df_exp.display()

###Arrray contains (returns boolean)


In [0]:
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()

###Group_by

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'),avg('Item_MRP')).display()

###Collect_list



In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()


In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

###Pivot

In [0]:
df.select('Item_Type','Outlet_Size','Item_MRP').display()

df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

###When_otherwise

In [0]:
df = df.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non-Veg').otherwise('Veg'))
    
df.display()

In [0]:
df.withColumn('veg_exp_flag',when(((col('veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                            .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'Veg_Expensive')\
                            .otherwise('Non_Veg')).display() 

###Joins

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()
df2.display()

#####inner join

In [0]:
df1.join(df2,df1.dept_id==df2.dept_id,'inner').display()

#####Left join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

#####Right join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()


#####Anti join

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

###Window Functions

#####Row_number()

In [0]:
from pyspark.sql.window import Window
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

Rank vs Dense Rank()

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
#Production practice...partitioning
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Partitions the data by category, allowing distributed parallel processing
window_spec = Window.partitionBy("Item_Type").orderBy(col("Item_MRP").desc())

df.withColumn("Rank_Per_Category", row_number().over(window_spec)).display()

###Cumulative sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display() #--SUMS every category no cumulative sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('totalsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display() ##Returns total of everything

###User defined functions(UDF)

In [0]:
def my_func(x):
    return x*x

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol',my_udf('Item_MRP')).display()

###Data writing

In [0]:
#csv
df.write.format('csv')\
        .save('/Volumes/workspace/default/data/Sales.csv')

#####Data writing modes

In [0]:
#append
df.write.format('csv')\
        .mode('append')\
        .save('/Volumes/workspace/default/data/Sales.csv')


In [0]:
df.write.format('csv')\
        .mode('append')\
        .option('path','/Volumes/workspace/default/data/Sales.csv')\
        .save()

In [0]:
#Overwrite
df.write.format('csv')\
        .mode('overwrite')\
        .option('path','/Volumes/workspace/default/data/Sales.csv')\
        .save()

In [0]:
#Error
df.write.format('csv')\
        .mode('error')\
        .option('path','/Volumes/workspace/default/data/Sales.csv')\
        .save()

In [0]:
#ignore
df.write.format('csv')\
        .mode('ignore')\
        .option('path','/Volumes/workspace/default/data/Sales.csv')\
        .save()

#####Parquet (columnar format)

In [0]:
#append
df.write.format('parquet')\
        .mode('append')\
        .option('path','/Volumes/workspace/default/data/Sales.csv')\
        .save()

In [0]:
#Overwrite
df.write.format('parquet')\
        .mode('overwrite')\
        .option('path','/Volumes/workspace/default/data/Sales.csv')\
        .save()

###Table(Managed and external tables
)

In [0]:
df.write.format('delta')\
.mode('overwrite')\
.saveAsTable('my_table')

###Spark SQL

#####Create a tenpView

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view

In [0]:
%sql
select * from my_view where Item_Fat_Content = 'Low Fat' or Item_Fat_Content = 'Lf'

In [0]:
df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'Low Fat' ")

In [0]:
df_sql.display()